In [ ]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1521

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


In [ ]:
#import pyspark; print(pyspark.__version__)
!pip uninstall pyspark

In [1]:
import os, sys, findspark

# 1. Setup paths
os.environ["SPARK_HOME"] = "C:/spark"
os.environ["HADOOP_HOME"] = "C:/hadoop-3.4.2"
os.environ["JAVA_HOME"] = "C:/Program Files/Java/jdk-21" # Verify this path!

# 2. Initialize
findspark.init()

# 3. Initialize findspark so it uses C:/spark
findspark.init()
print("Findspark initialized successfully.") 

Findspark initialized successfully.


In [2]:
from pyspark.sql import SparkSession
import time
 
# 1. Initialize Spark Session with Hive Support
# Replace 'path/to/mssql-jdbc.jar' with your actual jar location
# Use the local Windows backslash or triple slash for file URI
jar_path = "file:///C:/hadoop-3.4.2/mssql-jdbc-13.2.1.jre11.jar"
 
spark = SparkSession.builder \
    .appName("SQLServerToHive_2025") \
    .master("local[*]") \
    .config("spark.jars", jar_path) \
    .config("spark.driver.extraClassPath", jar_path) \
    .config("hive.metastore.uris", "thrift://localhost:9083") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark 3.5.6 is now active and connected to Hive!")
#print("Spark Session created successfully")



Spark 3.5.6 is now active and connected to Hive!


In [3]:
# 2. Configure SQL Server Connection
jdbc_url = "jdbc:sqlserver://localhost:1433;databaseName=sttwk_2025a_par;encrypt=true;trustServerCertificate=true"
db_properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}
start_time = time.time()
# 3. Read the View from SQL Server
# Note: Use the view name as the 'dbtable'
view_name = "sttwk_2022a"
df = spark.read.jdbc(url=jdbc_url, table=view_name, properties=db_properties)
# Optional: display schema to confirm data types
df.printSchema() 
record_count = df.count()
print(f"Total rows read from SQL Server: {record_count}")
end_time = time.time()
print(f"Data read complete. {record_count} rows read in {end_time - start_time:.2f} seconds.")


root
 |-- branch_id: string (nullable = true)
 |-- sp_id: string (nullable = true)
 |-- custd_id: string (nullable = true)
 |-- prod_id: string (nullable = true)
 |-- sls_id: string (nullable = true)
 |-- year: string (nullable = true)
 |-- month: string (nullable = true)
 |-- week: string (nullable = true)
 |-- net_qty: decimal(12,4) (nullable = true)
 |-- net_amt: decimal(18,4) (nullable = true)
 |-- grs_qty: decimal(12,4) (nullable = true)
 |-- grs_amt: decimal(18,4) (nullable = true)
 |-- rtn_qty: decimal(10,2) (nullable = true)
 |-- rtn_amt: decimal(15,2) (nullable = true)

Total rows read from SQL Server: 7142204
Data read complete. 7142204 rows read in 19.12 seconds.


In [10]:
# Save as a single CSV file for easy transport
df.write.mode("overwrite").option("header", "false").csv("C:/hadoop-3.4.2/sttwk_data")
print("Data saved to Windows folder!")


Data saved to Windows folder!


In [9]:
import time

# Use the standalone Hive JDBC driver we discussed
hive_jdbc_url = "jdbc:hive2://localhost:10000/adventuresales;auth=noSasl"

hive_properties = {
    "user": "root",
    "password": "",
    "driver": "org.apache.hive.jdbc.HiveDriver",
    "batchsize": "10000"
}

print(f"Starting transfer of {df.count()} rows via JDBC...")
start_write = time.time()

try:
    # This bypasses the 'get_table' protocol error
    df.write.jdbc(url=hive_jdbc_url, table="sttwk", mode="append", properties=hive_properties)
    
    duration = time.time() - start_write
    print(f"Data transfer complete in {duration:.2f} seconds!")
    
except Exception as e:
    print(f"Transfer failed: {e}")



Starting transfer of 7142204 rows via JDBC...
Transfer failed: An error occurred while calling o70.jdbc.
: java.sql.SQLException: Could not open client transport with JDBC Uri: jdbc:hive2://localhost:10000/adventuresales;auth=noSasl: Could not establish connection to jdbc:hive2://localhost:10000/adventuresales;auth=noSasl: null
	at org.apache.hive.jdbc.HiveConnection.<init>(HiveConnection.java:224)
	at org.apache.hive.jdbc.HiveDriver.connect(HiveDriver.java:107)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.BasicConnectionProvider.getConnection(BasicConnectionProvider.scala:49)
	at org.apache.spark.sql.execution.datasources.jdbc.connection.ConnectionProviderBase.create(ConnectionProvider.scala:102)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$1(JdbcDialects.scala:161)
	at org.apache.spark.sql.jdbc.JdbcDialect.$anonfun$createConnectionFactory$1$adapted(JdbcDialects.scala:157)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcRelationP

In [5]:
import time

print(f"Starting transfer of {df.count()} rows to Hive (adventuresales.sttwk)...")
start_write = time.time()

# Write to Hive inside Docker
try:
    # insertInto matches columns by position into the existing table
    df.write.mode("append").insertInto("adventuresales.sttwk")
    
    end_write = time.time()
    print("Data transfer complete!")
    print(f"Write duration: {end_write - start_write:.2f} seconds.")
    
except Exception as e:
    print(f"Transfer failed: {e}")
    print("Tip: Check if 'hdfs --daemon start namenode' and 'hive --service metastore' are still running in Docker.")


Starting transfer of 7142204 rows to Hive (adventuresales.sttwk)...
Transfer failed: org.apache.hadoop.hive.ql.metadata.HiveException: Unable to fetch table sttwk. Invalid method name: 'get_table'
Tip: Check if 'hdfs --daemon start namenode' and 'hive --service metastore' are still running in Docker.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat, col
import time

# Initialize Spark
spark = SparkSession.builder \
    .appName("AdventureSales_Migration") \
    .config("spark.sql.warehouse.dir", "C:/hadoop-3.4.2/warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

print(f"The active Spark version is: {spark.version}")
print("Spark session created successfully.")

# 1. SQL Server Connection Details
jdbc_url = "jdbc:sqlserver://localhost:1433;databaseName=sttwk_2025a_par;encrypt=false;trustServerCertificate=true"
connection_properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}
start_time = time.time()
# 2. Optimized Query (Filtering on SQL Server side is much faster)
sql_query = """(
    SELECT branch_id, sp_id, custd_id, prod_id, sls_id, 
           year, month, week, net_qty, net_amt, 
           grs_qty, grs_amt, rtn_qty, rtn_amt,
           year_month AS YearMonth 
    FROM sttwk 
    WHERE year_month < '202204'
) AS src"""

df = spark.read.jdbc(url=jdbc_url, table=sql_query, properties=connection_properties)

# Optional: display schema to confirm data types
df.printSchema() 
record_count = df.count()
print(f"Total rows read from SQL Server: {record_count}")
end_time = time.time()
print(f"Data read complete. {record_count} rows read in {end_time - start_time:.2f} seconds.")


In [ ]:
start_time = time.time()
# 3. Write to Hive (This will map YearMonth to the Hive Partition)
df.write \
    .mode("append") \
    .format("parquet") \
    .partitionBy("YearMonth") \
    .saveAsTable("AdventureSales.sttwk")

print("Migration for records before 2022-04 complete!")
end_time = time.time()
print(f"Data read complete. {record_count} rows read in {end_time - start_time:.2f} seconds.")